In [ ]:
from src.lm import load_GPT2, score_gpt2
from src.embedder import load_contriever, embed_sentences
from src.null_doc import NullDocumentEmbedding
from src.load_data import load_train_data, load_mmlu, divide_trainset
from src.index import rebuild_index, build_or_load_index
# from src.train_lsr import train_replug_lsr

In [ ]:
DEVICE = "cuda"
EMBEDDING_BS = 64
EMBEDDING_SIZE = 768
EVAL_SIZE = 1000

In [ ]:
train_queries, corpus_texts = load_train_data('yasnansh/all_queries_train', 'yasnansh/all_docs_train')
train_queries, eval_queries  = divide_trainset(train_queries, test_size=EVAL_SIZE)

mmlu_dev, mmlu_test = load_mmlu()

In [ ]:
# Load LM
lm, lm_tokenizer = load_GPT2('gpt2', "cuda")

In [ ]:
# Load retriever
emb_model, emb_tokenizer = load_contriever()

In [ ]:
def embed_fn(texts):
    return embed_sentences(texts, emb_tokenizer, emb_model, DEVICE, batch_size=EMBEDDING_BS)

def score_lm_fn(prefix, continuation):
    return score_gpt2(prefix, continuation, prefix, continuation, DEVICE)

null_doc_scorer = NullDocumentEmbedding(EMBEDDING_SIZE).to(DEVICE)

In [ ]:
# Load FAISS Index
global_index_ref = []
index_ref = build_or_load_index(corpus_texts, embed_fn, "replug_faiss.index")
global_index_ref.append(index_ref)
print(f"Index ready.  ntotal={index_ref.ntotal:,}")

In [ ]:
# Train REPLUG LSR TODO
def _rebuild_fn(texts):
    global_index_ref[0] = rebuild_index(texts, embed_fn)

config = {
    "lr": 2e-5,
    "warmup_steps": 100,
    "total_steps": 1000,
    "batch_size": 16,
    "top_k_train": 16,
    "gamma": 1.0,
    "index_refresh_every": 200,
    "index_path": "contriever_index.faiss",
}
# TODO
# lsr_losses = train_replug_lsr(
#     c_model        = emb_model,
#     c_tokenizer    = emb_tokenizer,
#     train_queries  = train_queries,
#     corpus_texts   = corpus_texts,
#     index_ref      = global_index_ref,
#     device         = DEVICE,
#     gpt2_tok       = lm_tokenizer,
#     gpt2_lm        = lm,
#     embed_fn       = embed_fn,
#     config         = config,
# )